# Семинар 09. Области видимости и декораторы

Разберём, как функция получает правило для поиска, где хранит состояние между вызовами и что происходит с результатом или исключением, когда вокруг функции появляется обёртка.

## Цели

- Передавать функцию как аргумент и решать, когда её вызвать.
- Сохранять настройку и изменяемое состояние в замыкании.
- Писать декоратор, который передаёт аргументы, возвращает результат и сохраняет сведения о функции.
- Различать независимое состояние обёрток, попадание в кэш и обращение к запасному источнику.
- Предсказывать порядок вызовов, когда на функции несколько декораторов.

## Перед началом

Сначала предскажите вывод трёх фрагментов разминки. Затем решите пять задач по порядку. Дополнительная задача использует декораторы из заданий 4 и 5. Теорию можно освежить в [лекции](lecture.ipynb).

В каждой заготовке замените `raise NotImplementedError` своим кодом. Запустите ячейку с определением, затем ячейку проверок под ней. До решения проверки должны падать. Для обёрток выполните ячейку с импортом `wraps` ниже.

Входные данные приходят через параметры. Возвращайте ответ через `return`, без `input` и `print`. Входные списки сохраняют содержимое. Состояние храните внутри созданных функций. Весь код работает в Python 3.10 и новее, включая Colab. Все данные для запуска есть в ноутбуке.

In [ ]:
from functools import wraps

## Разминка. Что напечатает код?

1. Какой префикс будет искать проверка внутри `choose`?
2. Сколько отдельных списков создаст фабрика? Какие имена обращаются к одному списку?
3. Когда появится первая запись в журнале декоратора? Что изменится после вызова `double`?

Запустите фрагменты и сверьтесь с разбором под ними.

In [ ]:
def make_prefix_filter(prefix):
    def matches(text):
        return text.startswith(prefix)
    return matches


def choose(rule):
    prefix = "архив:"
    return rule("отчёт: март"), rule("архив: март")


is_report = make_prefix_filter("отчёт:")
print(choose(is_report))

In [ ]:
def make_tray():
    rows = []

    def put(row):
        rows.append(row)
        return len(rows)

    return put


left = make_tray()
same = left
right = make_tray()
print(left("А"), same("Б"), right("В"))

In [ ]:
events = []


def watch(func):
    events.append("декорирование")

    @wraps(func)
    def wrapper(*args, **kwargs):
        events.append("вызов")
        return func(*args, **kwargs)

    def history():
        return events.copy()

    wrapper.history = history
    return wrapper


@watch
def double(x):
    return 2 * x


print(double.history())
print(double(x=3))
print(double.history())

<details>
<summary>Разбор разминки</summary>

**Фрагмент 1.** `matches` находит `prefix` во внешнем вызове `make_prefix_filter("отчёт:")`. У `choose` есть своё локальное имя `prefix`, доступное внутри `choose`.

```text
(True, False)
```

**Фрагмент 2.** `same` и `left` ссылаются на одну функцию. Вызов фабрики для `right` создал другой список. `append` меняет уже существующий список, поэтому `nonlocal` здесь не нужен.

```text
1 2 1
```

**Фрагмент 3.** Запись `@watch` вызывает `watch(double)` сразу после определения функции. Возвращённой обёртке мы добавили атрибут `history` со ссылкой на функцию. Поэтому `double.history()` показывает журнал, а `double(x=3)` запускает вычисление. В этом фрагменте журнал общий, в задаче 3 каждой обёртке понадобится свой.

```text
['декорирование']
6
['декорирование', 'вызов']
```

</details>

### Задание 1. Первая строка для проверки

В списке строк отчёта нужно найти первую, которую следует проверить вручную. Напишите `first_match(rows, rule)`. Функция `rule` принимает одну строку и возвращает `True` или `False`.

Верните индекс первой подходящей строки, считая с нуля. Если ни одна строка не подошла, верните `None`. Вызывайте правило по одному разу для каждой просмотренной строки, в исходном порядке. После первого совпадения остановитесь. На пустом списке правило ни разу не вызывается.

Пример. Для `["ок", "!сверить", "!подпись"]` и правила «начинается с `!`» ответ равен `1`. В списке до 100 000 строк.

In [ ]:
def first_match(rows: list[str], rule) -> int | None:
    raise NotImplementedError

In [ ]:
def check_first_match():
    visited = []

    def flagged(row):
        visited.append(row)
        return row.startswith("!")

    rows = ["ок", "!сверить", "!подпись"]
    assert first_match(rows, flagged) == 1
    assert visited == ["ок", "!сверить"]
    assert rows == ["ок", "!сверить", "!подпись"]
    visited.clear()
    assert first_match([], flagged) is None
    assert visited == []
    assert first_match(["!"], flagged) == 0
    assert first_match(["", "ок"], flagged) is None
    assert visited == ["!", "", "ок"]
    assert first_match(["да", ""], lambda row: row == "") == 1


check_first_match()

### Задание 2. Сигнал при пересечении порога

У двух филиалов свои пороги дневных продаж. Напишите `make_crossing(limit)`, которая возвращает функцию `observe(value)` для одного филиала.

`observe` возвращает `True` в тот день, когда значение впервые после периода ниже порога достигло `limit` или превысило его. В остальные дни ответ `False`. Перед первым наблюдением считайте, что филиал был ниже порога. После падения ниже порога сигнал снова возможен. Два вызова фабрики создают независимых наблюдателей, даже при одинаковом пороге.

Пример. При пороге `10` значения `[8, 10, 14, 9, 12]` дают `[False, True, False, False, True]`. Порог и значения целые, до 100 000 наблюдений. Для одного филиала храните постоянный объём данных.

In [ ]:
def make_crossing(limit: int):
    raise NotImplementedError

In [ ]:
def check_crossing():
    observe = make_crossing(10)
    assert [observe(x) for x in [8, 10, 14, 9, 12]] == [False, True, False, False, True]
    a = make_crossing(0)
    b = make_crossing(0)
    c = make_crossing(-2)
    assert a(0) is True
    assert b(4) is True
    assert a(0) is False
    assert c(-3) is False
    assert c(-2) is True
    assert b(-1) is False
    assert a(5) is False
    assert b(0) is True


check_crossing()

### Задание 3. Сводка по статусам

Функция обработки заявки возвращает строку со статусом. Напишите декоратор `tally_statuses`. Обёрнутая функция принимает прежние аргументы, вызывает исходную функцию ровно один раз и возвращает её результат. Каждый успешный вызов добавляет единицу к счётчику полученного статуса. Исключение исходной функции выходит наружу, счётчики при этом сохраняют свои значения.

У обёрнутой функции есть два дополнительных действия.

- `f.count(status)` возвращает число успешных вызовов с таким результатом. Для нового статуса ответ равен `0`.
- `f.reset(status)` обнуляет счётчик только этого статуса и возвращает `None`. Сброс нового статуса допустим.

Например, после результатов `"готово"`, `"брак"`, `"готово"` получим `f.count("готово") == 2`. После `f.reset("готово")` счётчик «брак» по-прежнему равен единице. У двух декорированных функций независимые сводки. Пустая строка тоже допустимый статус.

Сохраните имя и документацию исходной функции. Обёртка должна передавать позиционные и именованные аргументы. До 100 000 вызовов, запросов и сбросов. Каждый запрос счётчика должен работать в среднем за `O(1)` для коротких строк.

In [ ]:
def tally_statuses(func):
    raise NotImplementedError

In [ ]:
def check_tally():
    seen = []
    problem = ValueError("нет заявки")

    @tally_statuses
    def status(name, *, ok=True):
        """Вернуть статус заявки."""
        seen.append((name, ok))
        if name is None:
            raise problem
        return name if ok else "брак"

    @tally_statuses
    def other():
        return "готово"

    assert status.count("новый") == 0
    assert status.reset("новый") is None
    assert status("готово") == "готово"
    assert status("x", ok=False) == "брак"
    assert status(name="готово") == "готово"
    assert other.count("готово") == 0
    assert other() == "готово"
    assert status("") == ""
    try:
        status(None)
    except ValueError as error:
        assert error is problem
    else:
        raise AssertionError("Ожидалась ошибка исходной функции")
    assert seen == [("готово", True), ("x", False), ("готово", True), ("", True), (None, True)]
    assert status.count("готово") == 2
    assert status.count("") == 1
    assert status.reset("готово") is None
    assert (status.count("готово"), status.count("брак"), other.count("готово")) == (0, 1, 1)
    assert status("готово") == "готово"
    assert status.count("готово") == 1
    assert status.__name__ == "status" and status.__doc__ == "Вернуть статус заявки."


check_tally()

### Задание 4. Последняя котировка

Источник котировок принимает один строковый код и возвращает целую цену. Соседние строки отчёта часто относятся к одной валюте. Напишите декоратор `cache_last_quote`, который помнит только **одну** пару «код, цена».

Перед поиском в кэше удалите пробелы по краям кода и приведите буквы к верхнему регистру. При совпадении с сохранённым кодом сразу верните цену. Иначе вызовите источник с нормализованным кодом. Успешный результат заменяет прежнюю пару, включая цену `0`. Если источник поднял исключение, передайте его дальше и сохраните прежнюю пару.

Например, при вызовах с кодами `" usd "`, `"USD"`, `"eur"`, `"usd"` источник получит только `"USD"`, `"EUR"`, `"USD"`. После евро первая котировка доллара уже вытеснена.

Каждая обёртка хранит свою пару и сохраняет имя и документацию источника. Код состоит из латинских букв и пробелов по краям, после нормализации в нём от 1 до 12 букв. До 100 000 вызовов.

In [ ]:
def cache_last_quote(func):
    raise NotImplementedError

In [ ]:
def check_cache():
    calls = []
    missing = LookupError("пока нет котировки")

    @cache_last_quote
    def quote(code):
        """Цена по коду."""
        calls.append(code)
        if code == "GBP":
            raise missing
        return {"USD": 0, "EUR": 95}[code]

    @cache_last_quote
    def another(code):
        return 17

    assert quote(" usd ") == 0
    assert quote(code="Usd") == 0
    assert another("usd") == 17
    assert quote("USD") == 0
    for _ in range(2):
        try:
            quote("gbp")
        except LookupError as error:
            assert error is missing
        else:
            raise AssertionError("Ожидалось исключение источника")
    assert quote(" USD ") == 0
    assert quote(" eur") == 95
    assert quote("EUR ") == 95
    assert quote("USD") == 0
    assert calls == ["USD", "GBP", "GBP", "EUR", "USD"]
    assert quote.__name__ == "quote" and quote.__doc__ == "Цена по коду."


check_cache()

### Задание 5. Запасной источник

Старые записи бывают в нескольких форматах. Напишите фабрику декораторов `on_value_error(*backups)`. При каждом запросе обёртка сначала вызывает основную функцию. После `ValueError` или его подкласса попробуйте следующий источник. Запасные источники переданы при создании декоратора в порядке предпочтения. Каждому передавайте те же позиционные и именованные аргументы.

Верните первый обычный результат, в том числе `0` или `None`. Если все источники подняли `ValueError`, выпустите наружу последнее исключение. Любая другая ошибка сразу завершает поиск и выходит наружу. Сохраните имя и документацию основной функции. Создание декоратора и декорирование сами по себе источники не запускают.

Пример. Основная функция читает целые рубли, первая запасная умеет читать строку `"договорная"` и возвращает для неё `None`, вторая обращается к архиву цен. Для `"0"` хватит основного источника. Для `"договорная"` нужны первые два, до архива дело не дойдёт.

Запасных функций от нуля до десяти. Каждую используйте не более одного раза за запрос. При следующем запросе снова начните с основной.

In [ ]:
def on_value_error(*backups):
    raise NotImplementedError

In [ ]:
def check_backup():
    calls = []
    broken = TypeError("сбой источника")
    unknown = ValueError("неизвестный формат")

    class OldFormat(ValueError):
        pass

    def legacy(text, *, factor=1):
        calls.append(("запас", text, factor))
        if text == "договорная":
            return None
        if text == "сбой":
            raise broken
        raise ValueError(text)

    def archive(text, *, factor=1):
        calls.append(("архив", text, factor))
        if text == "нет":
            raise unknown
        return 7 * factor

    @on_value_error(legacy, archive)
    def main(text, *, factor=1):
        """Основной источник."""
        calls.append(("основной", text, factor))
        if text == "тип":
            raise broken
        if text == "0":
            return 0
        raise OldFormat(text)

    assert calls == []
    for text, value, route in [
        ("0", 0, ["основной"]),
        ("договорная", None, ["основной", "запас"]),
        ("архивная", 14, ["основной", "запас", "архив"]),
    ]:
        calls.clear()
        assert main(text, factor=2) == value
        assert calls == [(name, text, 2) for name in route]
    for text, problem, route in [
        ("тип", broken, ["основной"]),
        ("сбой", broken, ["основной", "запас"]),
        ("нет", unknown, ["основной", "запас", "архив"]),
    ]:
        calls.clear()
        try:
            main(text=text)
        except (ValueError, TypeError) as error:
            assert error is problem
        else:
            raise AssertionError("Ожидалась ошибка источника")
        assert calls == [(name, text, 1) for name in route]
    assert main.__name__ == "main" and main.__doc__ == "Основной источник."
    assert on_value_error()(int)("42") == 42


check_backup()

## Дополнительно. Когда запоминать запасную цену

Используйте готовые `cache_last_quote` и `on_value_error` из заданий 4 и 5. Напишите `make_quote_reader(primary, backup, *, cache_backup)`, которая возвращает функцию чтения цены. Оба источника принимают один код и возвращают целую цену. В этой задаче на вход подают уже нормализованные коды.

- Если `cache_backup=True`, запоминайте цену, полученную из любого источника.
- Если `cache_backup=False`, запоминайте только цену из основного. После выдачи запасной цены следующий запрос того же кода должен снова обратиться к основному источнику.

Соберите каждую версию наложением двух декораторов. В обёртках уже есть и кэш одной записи, и правила обработки ошибок. У каждого нового читателя свой кэш.

Пример. При первом запросе `"USD"` основной источник поднимает `ValueError`, а запасной даёт `0`. При следующем обращении основной уже готов вернуть `90`. Для двух одинаковых запросов первый режим даст `[0, 0]`, второй — `[0, 90]`. До 100 000 запросов.

In [ ]:
def make_quote_reader(primary, backup, *, cache_backup: bool):
    raise NotImplementedError

In [ ]:
def check_reader():
    for cache_backup, expected, sources in [
        (True, [0, 0, 0], ["основной", "запас"]),
        (False, [0, 90, 90], ["основной", "запас", "основной"]),
    ]:
        calls = []

        def primary(code):
            calls.append(("основной", code))
            if len(calls) == 1:
                raise ValueError("курс ещё не загружен")
            return 90

        def backup(code):
            calls.append(("запас", code))
            return 0

        read = make_quote_reader(primary, backup, cache_backup=cache_backup)
        assert calls == []
        assert [read("USD") for _ in range(3)] == expected
        assert calls == [(name, "USD") for name in sources]
        fresh = make_quote_reader(primary, backup, cache_backup=cache_backup)
        assert fresh("USD") == 90
        assert calls[-1] == ("основной", "USD")
        assert len(calls) == len(sources) + 1


check_reader()

## Итоги

- Правило поиска получает очередную строку, пока не найдено первое совпадение.
- Настройка и состояние живут в замыкании. Два вызова фабрики создают две независимые истории.
- Декоратор решает, когда вызвать исходную функцию, что вернуть и какие ошибки обработать. `wraps` сохраняет сведения о функции.
- При нескольких декораторах запрос входит во внешнюю обёртку первым. Обычный результат или исключение проходит обратно через те обёртки, которые уже начали работу.